# Electricity Model Validation and Forecast Artifact Audit

**Role**  
Independent validation of frozen Electricity forecast artifacts before downstream trustworthiness analysis.

**Audit Scope**  
Protocol A and Protocol B forecast integrity, alignment, reconstruction, metric reproducibility, and artifact-level diagnostics.

**Inputs**  
The authoritative input inventory is built dynamically in Section 4 from the validated Protocol A and B forecasts, Protocol B horizon metrics, classical-model selection metadata, native uncertainty summary, and canonical Electricity TSF data.

**Outputs**  
Validation evidence displayed in this notebook. No audit artifact or forecast file is written.

**Depends On**  
`10_Electricity_EDA.ipynb` · `11_Electricity_Classical_Baselines.ipynb` · `12_Electricity_LSTM.ipynb` · `13_Electricity_Foundation_Models.ipynb`

**Authoritative Status**  
Upstream notebooks generate model forecasts. Notebook 14 independently certifies the frozen evidence boundary used downstream.

**What This Notebook Does Not Do**

- train models, regenerate forecasts, or select hyperparameters;
- evaluate robustness or full uncertainty;
- compute Trust Scores or perform statistical significance testing;
- prove the absence of every possible upstream implementation error.


## 1. Objective

**Central validation question.** Do the frozen Electricity forecast artifacts satisfy the structural, temporal, information-set, reconstruction, and metric-consistency requirements necessary for them to serve as trustworthy inputs to downstream robustness, uncertainty, trustworthiness, and statistical analyses?

Subquestions:

1. Are all forecast artifacts structurally complete and finite?
2. Are forecasts aligned exactly with their intended targets?
3. Is Protocol A genuinely rolling one-step?
4. Is Protocol B genuinely 48-step day-ahead without within-horizon actual updates?
5. Can deterministic baseline forecasts be independently reconstructed?
6. Are saved model vectors consistent with their documented update discipline?
7. Can reported metrics be independently reproduced from frozen vectors?
8. Do basic distribution diagnostics reveal collapsed, duplicated, or otherwise suspicious predictions?


## 2. Setup


In [ ]:
from pathlib import Path
import hashlib, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

def find_project_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate/'src').is_dir() and (candidate/'results/electricity').is_dir(): return candidate
    raise FileNotFoundError('Could not locate project root.')

ROOT=find_project_root(Path.cwd()); R=ROOT/'results/electricity'; DATA=ROOT/'data/electricity/australian_electricity_demand_dataset.tsf'
MODELS=['Naive','Daily_Seasonal_Naive','Weekly_Seasonal_Naive','Moving_Average','ARIMA','SARIMA','Prophet','Simple_Exponential_Smoothing','Holt_Winters','DHR_ARIMA','LSTM','Chronos_Bolt_Tiny','TimesFM']
BASELINES=MODELS[:4]; CLASSICAL=['ARIMA','SARIMA','Prophet','Simple_Exponential_Smoothing','Holt_Winters']; TOL=1e-12; SEASONAL_LAG=48
plt.style.use('seaborn-v0_8-whitegrid'); plt.rcParams.update({'figure.figsize':(10,4),'axes.titlesize':12})

def metrics(a,p,scale):
    a=np.asarray(a,float); p=np.asarray(p,float); e=a-p
    return {'MAE':np.mean(abs(e)),'RMSE':np.sqrt(np.mean(e**2)),'MAPE':np.mean(abs(e/a))*100,'sMAPE':np.mean(2*abs(e)/(abs(a)+abs(p)))*100,'MASE_48':np.mean(abs(e))/scale}
def audit_row(category,check,protocol='Both',subject='—',expected='—',observed='—',status='PASS'):
    return {'Category':category,'Check':check,'Protocol':protocol,'Model / Artifact':subject,'Expected':expected,'Observed':observed,'Status':status}
def sha256(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()
def show(frame,digits=6): display(frame.round(digits))
audit_rows=[]


In [ ]:
def load_tsf(path):
    attrs=[]; rows=[]
    with Path(path).open(encoding='utf-8') as handle:
        for raw in handle:
            line=raw.strip()
            if not line or line.startswith('#'): continue
            if line.startswith('@attribute'):
                _,name,kind=line.split(maxsplit=2); attrs.append((name,kind))
            elif not line.startswith('@'):
                parts=line.split(':',len(attrs)); row=dict(zip((a[0] for a in attrs),parts[:-1])); row['series_value']=np.fromstring(parts[-1],sep=','); rows.append(row)
    return pd.DataFrame(rows)
raw=load_tsf(DATA); selected=raw[(raw.series_name=='T4')&(raw.state=='SA')]; assert len(selected)==1
source=selected.iloc[0]; source_index=pd.date_range(pd.to_datetime(source.start_timestamp,format='%Y-%m-%d %H-%M-%S'),periods=len(source.series_value),freq='30min')
y=pd.Series(source.series_value,index=source_index,name='Actual'); pretest=y.loc[:'2012-07-12 23:30']; test=y.loc['2012-07-13':'2015-03-01 23:30']
scale48=float(np.mean(np.abs(pretest.to_numpy()[SEASONAL_LAG:]-pretest.to_numpy()[:-SEASONAL_LAG])))


## 3. Audit Framework and Certification Criteria

### 3.1 Frozen Evidence Boundary

A frozen forecast artifact is a preserved target-aligned forecast vector produced upstream and treated as immutable downstream evidence. Auditing preserved vectors rather than silently rerunning models supports reproducibility and fair comparison, separates expensive generation from cheap analysis, protects against post-hoc changes, and prevents downstream result drift.

### 3.2 Validation Dimensions


In [ ]:
validation_dimensions=pd.DataFrame([
['Schema integrity','Are required fields and model columns present?','Malformed or incomplete evidence.'],['Timestamp integrity','Does every forecast have the intended ordered timestamp?','Broken join, off-by-one error, or leakage risk.'],['Target alignment','Do stored actuals equal the canonical target?','Wrong evaluation target or shifted forecast.'],['Forecast horizon','Are all Protocol B origins complete at horizons 1–48?','Truncated, duplicated, or stitched day-ahead evidence.'],['Information-set discipline','Is behavior consistent with the documented update rule?','Potential protocol violation; internals may remain unobservable.'],['Finiteness/missingness','Are all actuals and forecasts finite and present?','Invalid metric inputs.'],['Independent baseline reconstruction','Do manual rules reproduce frozen vectors row by row?','Generator inconsistency or alignment error.'],['Model-vector identity','Are unrelated model vectors unexpectedly identical?','Possible filename/column substitution.'],['Metric reproducibility','Do canonical metrics reproduce from frozen vectors?','Summary/vector inconsistency.'],['Forecast-distribution plausibility','Are forecasts nonconstant and on a plausible scale?','Collapse, instability, or gross mismatch.'],['Cross-protocol discipline','Are protocols compared within their information sets?','Scientifically invalid combined interpretation.']],columns=['Validation Dimension','Question','Failure Would Indicate'])
display(validation_dimensions)


### 3.3 Certification Levels

- **Structural validation:** artifact shape, schema, timestamps, ordering, finiteness, and horizon coverage.
- **Behavioral / implementation validation:** independent reconstruction, update-discipline relationships, vector distinctness, and horizon discipline.
- **Numerical reproducibility:** canonical metrics recomputed directly from frozen vectors.

A PASS establishes internal evidence consistency under the stated check. It does not mathematically prove that every upstream model implementation is correct. Questions invisible in final vectors are labelled **NOT DIRECTLY TESTABLE** rather than promoted to PASS.


## 4. Authoritative Artifact Inventory

### 4.1 Artifact Inventory — Table


In [ ]:
artifact_spec=[
('protocol_a_validated_forecasts.csv','A','Validated actuals and all point forecasts'),('protocol_b_validated_forecasts.csv','B','Validated day-ahead actuals and point forecasts'),('protocol_b_validated_horizon_metrics.csv','B','Frozen horizon-level metric reference'),('classical_model_selection.json','Both','Classical selection and update metadata'),('uncertainty_summary.csv','Both','Native uncertainty availability evidence')]
loaded={}; inventory=[]
for name,protocol,purpose in artifact_spec:
    path=R/name; exists=path.exists(); assert exists
    if path.suffix=='.csv':
        frame=pd.read_csv(path,parse_dates=[x for x in ['Origin','Timestamp'] if x in pd.read_csv(path,nrows=0).columns]); loaded[name]=frame
        ts=frame.Timestamp if 'Timestamp' in frame else None
        inventory.append({'Artifact':name,'Protocol':protocol,'Purpose':purpose,'Rows':len(frame),'Columns / Model Count':len(frame.columns),'Timestamp Start':ts.min() if ts is not None else pd.NaT,'Timestamp End':ts.max() if ts is not None else pd.NaT,'Status':'LOADED'})
    else:
        loaded[name]=json.loads(path.read_text(encoding='utf-8')); inventory.append({'Artifact':name,'Protocol':protocol,'Purpose':purpose,'Rows':len(loaded[name]),'Columns / Model Count':'JSON metadata','Timestamp Start':pd.NaT,'Timestamp End':pd.NaT,'Status':'LOADED'})
artifact_inventory=pd.DataFrame(inventory); display(artifact_inventory)
pa=loaded['protocol_a_validated_forecasts.csv']; pb=loaded['protocol_b_validated_forecasts.csv']; hm=loaded['protocol_b_validated_horizon_metrics.csv']; selection_metadata=loaded['classical_model_selection.json']; uncertainty=loaded['uncertainty_summary.csv']


### 4.2 Model Coverage — Table


In [ ]:
coverage=pd.DataFrame([{'Model':m,'Protocol A Artifact Available?':m in pa,'Protocol B Artifact Available?':m in pb,'Expected?':True,'Audit Scope':'Structural, numerical, distribution'+(' + exact reconstruction' if m in BASELINES else '')} for m in MODELS])
display(coverage)


### 4.3 Loading Integrity


In [ ]:
loading=[]
for name,protocol,_ in artifact_spec:
    obj=loaded[name]; schema_ok=True; duplicate_columns=False; duplicate_index=False
    if isinstance(obj,pd.DataFrame):
        duplicate_columns=obj.columns.duplicated().any(); schema_ok=not duplicate_columns
        if 'Timestamp' in obj: duplicate_index=obj.Timestamp.duplicated().any()
    status='PASS' if schema_ok and not duplicate_columns and (name!='protocol_a_validated_forecasts.csv' or not duplicate_index) else 'FAIL'
    loading.append({'Artifact':name,'File exists':True,'Read succeeds':True,'Expected schema present':schema_ok,'Duplicate columns':duplicate_columns,'Unexpected index duplication':duplicate_index,'Status':status})
    audit_rows.append(audit_row('Artifact Loading','Load and schema',protocol,name,'Readable, unique schema',status,status))
loading_integrity=pd.DataFrame(loading); display(loading_integrity); assert loading_integrity.Status.eq('PASS').all()


## 5. Protocol A Validation — Rolling One-Step

### 5.1 Protocol Definition

Protocol A follows: forecast `t` → observe actual `t` → actual `t` may become available for forecast `t+1`. It is rolling one-step operation, not a static multi-step forecast.

### 5.2 Structural Integrity — Table


In [ ]:
expected_a_columns=['Timestamp','Actual']+MODELS
a_checks=[('Expected row count',len(test),len(pa),len(pa)==len(test)),('Timestamps unique',True,pa.Timestamp.is_unique,pa.Timestamp.is_unique),('Timestamps sorted',True,pa.Timestamp.is_monotonic_increasing,pa.Timestamp.is_monotonic_increasing),('Target alignment',True,np.array_equal(pa.Actual,test.to_numpy()),np.array_equal(pa.Actual,test.to_numpy())),('All actuals finite',True,np.isfinite(pa.Actual).all(),np.isfinite(pa.Actual).all()),('All forecasts finite',True,np.isfinite(pa[MODELS].to_numpy()).all(),np.isfinite(pa[MODELS].to_numpy()).all()),('No missing forecasts',True,not pa[MODELS].isna().any().any(),not pa[MODELS].isna().any().any()),('Expected model columns present',True,set(expected_a_columns).issubset(pa.columns),set(expected_a_columns).issubset(pa.columns))]
protocol_a_structural=pd.DataFrame([{'Check':n,'Expected':e,'Observed':o,'Status':'PASS' if ok else 'FAIL'} for n,e,o,ok in a_checks]); display(protocol_a_structural)
for n,e,o,ok in a_checks: audit_rows.append(audit_row('Schema Integrity' if 'column' in n.lower() or 'row' in n.lower() else 'Protocol A Integrity',n,'A','Validated forecast',e,o,'PASS' if ok else 'FAIL'))
assert protocol_a_structural.Status.eq('PASS').all()


### 5.3 Timestamp Alignment — Table


In [ ]:
expected_ts=pd.Series(test.index,name='Timestamp'); mismatch_ns=np.abs((pa.Timestamp-expected_ts).dt.total_seconds()).max()
a_timestamp=pd.DataFrame([{'First forecast timestamp':pa.Timestamp.iloc[0],'Last forecast timestamp':pa.Timestamp.iloc[-1],'Expected first timestamp':expected_ts.iloc[0],'Expected last timestamp':expected_ts.iloc[-1],'Maximum timestamp mismatch (s)':mismatch_ns,'Status':'PASS' if mismatch_ns==0 else 'FAIL'}]); display(a_timestamp)
audit_rows.append(audit_row('Timestamp Integrity','Exact target timestamps','A','Validated forecast','0 s mismatch',mismatch_ns,a_timestamp.Status.iloc[0])); assert mismatch_ns==0


### 5.4 Information-Set Guardrail

The artifact directly proves target alignment, completeness, and the one-value-per-target representation. Exact reconstruction supplies stronger evidence for deterministic baselines. For fitted, neural, and foundation models, the final vector cannot by itself prove which observations entered upstream computation; their internal no-leakage discipline remains **not directly testable from frozen artifacts**.


## 6. Protocol B Validation — 48-Step Day-Ahead

### 6.1 Protocol Definition

Each origin produces 48 half-hour predictions without using actual values revealed inside that horizon.

### 6.2 Structural Integrity — Table


In [ ]:
origin_groups=pb.groupby('Origin',sort=True); expected_origins=len(test)//48
b_checks=[('Number of origins',expected_origins,pb.Origin.nunique(),pb.Origin.nunique()==expected_origins),('48 horizons per origin',True,origin_groups.size().eq(48).all(),origin_groups.size().eq(48).all()),('Origins sorted',True,pb.Origin.is_monotonic_increasing,pb.Origin.is_monotonic_increasing),('Target timestamps aligned',True,pb.Timestamp.equals(expected_ts),pb.Timestamp.equals(expected_ts)),('Origin/horizon pairs unique',True,not pb.duplicated(['Origin','Horizon']).any(),not pb.duplicated(['Origin','Horizon']).any()),('No missing horizon',True,origin_groups.Horizon.apply(lambda s:s.tolist()==list(range(1,49))).all(),origin_groups.Horizon.apply(lambda s:s.tolist()==list(range(1,49))).all()),('Actuals finite',True,np.isfinite(pb.Actual).all(),np.isfinite(pb.Actual).all()),('Forecasts finite',True,np.isfinite(pb[MODELS].to_numpy()).all(),np.isfinite(pb[MODELS].to_numpy()).all()),('Expected model columns',True,set(['Origin','Timestamp','Horizon','Actual']+MODELS).issubset(pb.columns),set(['Origin','Timestamp','Horizon','Actual']+MODELS).issubset(pb.columns))]
protocol_b_structural=pd.DataFrame([{'Check':n,'Expected':e,'Observed':o,'Status':'PASS' if ok else 'FAIL'} for n,e,o,ok in b_checks]); display(protocol_b_structural)
for n,e,o,ok in b_checks: audit_rows.append(audit_row('Horizon Integrity' if 'horizon' in n.lower() or 'origin' in n.lower() else 'Protocol B Integrity',n,'B','Validated forecast',e,o,'PASS' if ok else 'FAIL'))
assert protocol_b_structural.Status.eq('PASS').all()


### 6.3 Horizon Completeness — Visualization


In [ ]:
horizon_coverage=pb.groupby('Horizon').agg(Valid_predictions=('Actual','count'),Origins=('Origin','nunique')).reset_index(); expected_per_horizon=pb.Origin.nunique()
ax=horizon_coverage.plot.bar(x='Horizon',y='Valid_predictions',legend=False,color='#0072B2',figsize=(11,3)); ax.axhline(expected_per_horizon,color='black',linestyle='--',linewidth=.8); ax.set(title='Protocol B valid prediction count at every horizon',xlabel='Half-hour horizon',ylabel='Valid predictions'); plt.tight_layout(); plt.show()
assert horizon_coverage.Valid_predictions.eq(expected_per_horizon).all()


### 6.4 Within-Horizon Update Guardrail

Origin/horizon structure proves that each saved day is represented as a 48-step block. Deterministic baseline reconstruction can test the implied fixed-origin rule directly. For fitted and opaque models, a frozen vector cannot reveal internal context updates; absence of within-horizon actual access is therefore documented upstream but **not directly testable from the artifact alone**.


## 7. Independent Baseline Reconstruction

### 7.1 Why Independent Reconstruction Matters

Length checks are weak evidence. Re-deriving deterministic rules from the canonical target and comparing every value against the stored vectors directly tests generator consistency and alignment.

### 7.2 Reconstructed Baselines — Table


In [ ]:
expected_a={'Naive':y.shift(1).loc[test.index].to_numpy(),'Daily_Seasonal_Naive':y.shift(48).loc[test.index].to_numpy(),'Weekly_Seasonal_Naive':y.shift(336).loc[test.index].to_numpy(),'Moving_Average':y.rolling(48).mean().shift(1).loc[test.index].to_numpy()}
expected_b={m:np.empty(len(pb)) for m in BASELINES}; vals=y.to_numpy(); positions=pd.Series(np.arange(len(y)),index=y.index)
for _,g in pb.groupby('Origin',sort=True):
    origin=g.Origin.iloc[0]; p=int(positions.loc[origin]); ix=g.index.to_numpy()
    expected_b['Naive'][ix]=np.repeat(vals[p-1],48); expected_b['Daily_Seasonal_Naive'][ix]=vals[p-48:p]; expected_b['Weekly_Seasonal_Naive'][ix]=vals[p-336:p-288]
    history=list(vals[p-48:p].astype(float)); out=[]
    for _ in range(48): pred=float(np.mean(history[-48:])); out.append(pred); history.append(pred)
    expected_b['Moving_Average'][ix]=out
rules={'Naive':'last legally available observation','Daily_Seasonal_Naive':'lag 48','Weekly_Seasonal_Naive':'lag 336','Moving_Average':'recursive 48-value mean'}
recon=[]
for protocol,frame,expected in [('A',pa,expected_a),('B',pb,expected_b)]:
    for m in BASELINES:
        diff=np.max(np.abs(frame[m].to_numpy()-expected[m])); status='PASS' if diff<=TOL else 'FAIL'
        recon.append({'Model':m,'Protocol':protocol,'Reconstruction rule':rules[m],'Rows compared':len(frame),'Maximum absolute difference':diff,'Exact / tolerance match':diff<=TOL,'Status':status})
        audit_rows.append(audit_row('Baseline Reconstruction','Independent row-wise reconstruction',protocol,m,f'≤ {TOL}',diff,status))
reconstruction=pd.DataFrame(recon); show(reconstruction,14); assert reconstruction.Status.eq('PASS').all()


### 7.3 Boundary Examples — Table


In [ ]:
examples=[]
for protocol,frame,expected in [('A',pa,expected_a),('B',pb,expected_b)]:
    for m in BASELINES:
        for i in [0,1,len(frame)-2,len(frame)-1]: examples.append({'Protocol':protocol,'Model':m,'Timestamp':frame.Timestamp.iloc[i],'Actual':frame.Actual.iloc[i],'Expected source observation / rule':rules[m],'Manual reconstruction':expected[m][i],'Frozen forecast':frame[m].iloc[i],'Absolute difference':abs(expected[m][i]-frame[m].iloc[i])})
boundary_examples=pd.DataFrame(examples); show(boundary_examples)


### 7.4 Reconstruction Summary


In [ ]:
reconstruction_summary=reconstruction.pivot(index='Model',columns='Protocol',values='Maximum absolute difference'); show(reconstruction_summary,14)


## 8. Statistical and Classical Model Artifact Audit

### 8.1 Model Update Discipline — Table


In [ ]:
discipline=[]
family={'ARIMA':'ARIMA','SARIMA':'Seasonal ARIMA','Prophet':'Additive regression','Simple_Exponential_Smoothing':'Exponential smoothing','Holt_Winters':'Seasonal exponential smoothing','DHR_ARIMA':'Dynamic harmonic regression + ARIMA'}
for m in CLASSICAL:
    update=selection_metadata['Models'][m]['Update']; periodic=update.startswith('Validation-selected periodic refit')
    discipline.append({'Model':m,'Model family':family[m],'Protocol A update discipline':'Sequential state update' if not periodic else 'Periodic refit schedule','Protocol B update discipline':'Fixed-origin between refits' if periodic else 'No within-day actual update','Refit / state-update behavior':update,'Expected A/B equality?':'Yes' if periodic else 'No','Audit implication':'Equality is design-consistent' if periodic else 'Vectors should differ'})
discipline.append({'Model':'DHR_ARIMA','Model family':family['DHR_ARIMA'],'Protocol A update discipline':'Saved sequential forecast','Protocol B update discipline':'Saved 48-step forecast','Refit / state-update behavior':'No regeneration in audit','Expected A/B equality?':'No','Audit implication':'Structural evidence only'})
update_discipline=pd.DataFrame(discipline); display(update_discipline)


### 8.2 Artifact Integrity — Table


In [ ]:
classical_models=CLASSICAL+['DHR_ARIMA']; rows=[]
for protocol,frame in [('A',pa),('B',pb)]:
    for m in classical_models:
        ok=len(frame)==len(test) and frame.Timestamp.equals(expected_ts) and np.isfinite(frame[m]).all() and frame[m].var()>0
        rows.append({'Model':m,'Protocol':protocol,'Length':len(frame),'Aligned':frame.Timestamp.equals(expected_ts),'Finite':np.isfinite(frame[m]).all(),'Variance > 0':frame[m].var()>0,'Status':'PASS' if ok else 'FAIL'})
        audit_rows.append(audit_row('Classical Artifact','Length, alignment, finiteness, variance',protocol,m,'Valid',ok,'PASS' if ok else 'FAIL'))
classical_artifact_audit=pd.DataFrame(rows); display(classical_artifact_audit); assert classical_artifact_audit.Status.eq('PASS').all()


### 8.3 Expected Cross-Protocol Relationships


In [ ]:
relationship=[]
for m in classical_models:
    expected='Equal' if m in {'Prophet','Simple_Exponential_Smoothing','Holt_Winters'} else 'Different'
    diff=float(np.max(np.abs(pa[m].to_numpy()-pb[m].to_numpy()))); consistent=(diff<=TOL) if expected=='Equal' else (diff>TOL)
    status='EXPECTED / DESIGN-CONSISTENT' if consistent and expected=='Equal' else ('PASS' if consistent else 'FAIL')
    relationship.append({'Model':m,'Expected Relationship':expected,'Observed Max Absolute Difference':diff,'Consistent?':consistent,'Status':status,'Interpretation':'Expected by periodic-refit construction' if expected=='Equal' else 'Protocol-dependent vector observed'})
    audit_rows.append(audit_row('Update Discipline','Cross-protocol relationship','Both',m,expected,diff,status))
cross_protocol_relationships=pd.DataFrame(relationship); show(cross_protocol_relationships,14); assert cross_protocol_relationships['Consistent?'].all()


## 9. LSTM Artifact Audit

### 9.1 Expected Artifact Properties

The supervised LSTM has one aligned Protocol A value per target and one direct 48-value Protocol B block per origin. This audit does not retrain it or infer hidden training behavior from the saved vector.

### 9.2 Structural Audit — Table


In [ ]:
lstm=[]
for protocol,frame in [('A',pa),('B',pb)]:
    ok=len(frame)==len(test) and frame.Timestamp.equals(expected_ts) and np.isfinite(frame.LSTM).all() and not frame.LSTM.isna().any()
    lstm.append({'Protocol':protocol,'Expected N':len(test),'Observed N':len(frame),'Aligned?':frame.Timestamp.equals(expected_ts),'Finite?':np.isfinite(frame.LSTM).all(),'Missing?':frame.LSTM.isna().any(),'Forecast variance':frame.LSTM.var(),'Status':'PASS' if ok else 'FAIL'})
    audit_rows.append(audit_row('LSTM Artifact','Structural vector audit',protocol,'LSTM','Valid',ok,'PASS' if ok else 'FAIL'))
lstm_audit=pd.DataFrame(lstm); show(lstm_audit); assert lstm_audit.Status.eq('PASS').all()


### 9.3 Neural Forecast Sanity Diagnostics


In [ ]:
def sanity(frame,m):
    a=frame.Actual.to_numpy(float); p=frame[m].to_numpy(float); return {'Prediction std / Actual std':p.std(ddof=1)/a.std(ddof=1),'Forecast range / Actual range':np.ptp(p)/np.ptp(a),'Correlation with actual':np.corrcoef(a,p)[0,1],'Mean residual':np.mean(a-p)}
lstm_sanity=pd.DataFrame([{'Protocol':p,**sanity(f,'LSTM')} for p,f in [('A',pa),('B',pb)]]); show(lstm_sanity)


## 10. Foundation Model Artifact Audit

### 10.1 Foundation Artifact Inventory — Table


In [ ]:
foundation_info=[]
for m,checkpoint in [('Chronos_Bolt_Tiny','amazon/chronos-bolt-tiny'),('TimesFM','google/timesfm-2.5-200m-pytorch')]:
    u=uncertainty[(uncertainty.Model==m)&uncertainty.Available]
    foundation_info.append({'Model':m,'Checkpoint reference':checkpoint,'Protocol A available?':m in pa,'Protocol B available?':m in pb,'Native uncertainty artifact available?':not u.empty,'Point forecast finite?':np.isfinite(pa[m]).all() and np.isfinite(pb[m]).all(),'Status':'PASS'})
foundation_inventory=pd.DataFrame(foundation_info); display(foundation_inventory)


### 10.2 Structural Audit — Table


In [ ]:
foundation=[]
for protocol,frame in [('A',pa),('B',pb)]:
    for m in ['Chronos_Bolt_Tiny','TimesFM']:
        ok=len(frame)==len(test) and frame.Timestamp.equals(expected_ts) and not frame[m].isna().any() and np.isfinite(frame[m]).all() and frame[m].var()>0
        foundation.append({'Model':m,'Protocol':protocol,'Forecast count':len(frame),'Aligned':frame.Timestamp.equals(expected_ts),'Missing predictions':frame[m].isna().sum(),'Finite predictions':np.isfinite(frame[m]).all(),'Forecast variance':frame[m].var(),'Constant forecast?':frame[m].nunique()<=1,'Status':'PASS' if ok else 'FAIL'})
        audit_rows.append(audit_row('Foundation Artifact','Structural vector audit',protocol,m,'Valid',ok,'PASS' if ok else 'FAIL'))
foundation_audit=pd.DataFrame(foundation); show(foundation_audit); assert foundation_audit.Status.eq('PASS').all()


### 10.3 Forecast Sanity Diagnostics


In [ ]:
foundation_sanity=pd.DataFrame([{'Protocol':p,'Model':m,**sanity(f,m),'Change std ratio':np.diff(f[m]).std(ddof=1)/np.diff(f.Actual).std(ddof=1)} for p,f in [('A',pa),('B',pb)] for m in ['Chronos_Bolt_Tiny','TimesFM']]); show(foundation_sanity)


### 10.4 Foundation Forecast Diagnostic — Visualization

The first complete test week is selected before inspection. The plots audit shift, scale, and collapse; they are not ranking figures.


In [ ]:
segment=slice(0,7*48); fig,axes=plt.subplots(2,1,figsize=(11,7),sharex=True)
for ax,frame,protocol in [(axes[0],pa,'A'),(axes[1],pb,'B')]:
    ax.plot(frame.Timestamp.iloc[segment],frame.Actual.iloc[segment],label='Actual',color='black',linewidth=1.3)
    ax.plot(frame.Timestamp.iloc[segment],frame.Chronos_Bolt_Tiny.iloc[segment],label='Chronos',linewidth=.9)
    ax.plot(frame.Timestamp.iloc[segment],frame.TimesFM.iloc[segment],label='TimesFM',linewidth=.9)
    ax.set(title=f'Protocol {protocol}: foundation-artifact audit — first test week',ylabel='Demand (MW)'); ax.legend()
axes[-1].set_xlabel('Timestamp'); plt.tight_layout(); plt.show()


## 11. Cross-Model Distinctness Audit

### 11.1 Pairwise Exact-Identity Check — Table

Exact or near-exact identity can reveal accidental substitution. High correlation alone is not a failure.


In [ ]:
pairs=[(a,b) for i,a in enumerate(MODELS) for b in MODELS[i+1:]]; distinct=[]
for protocol,frame in [('A',pa),('B',pb)]:
    identical=[(a,b) for a,b in pairs if np.array_equal(frame[a].to_numpy(),frame[b].to_numpy())]
    near=[(a,b) for a,b in pairs if not np.array_equal(frame[a].to_numpy(),frame[b].to_numpy()) and np.max(np.abs(frame[a]-frame[b]))<=TOL]
    status='PASS' if not identical and not near else 'FAIL'
    distinct.append({'Protocol':protocol,'Model pairs tested':len(pairs),'Unexpected identical pairs':len(identical)+len(near),'Expected identical pairs':0,'Status':status})
    audit_rows.append(audit_row('Cross-Model Distinctness','Unexpected exact/near identity',protocol,'All model pairs','0',len(identical)+len(near),status))
distinctness=pd.DataFrame(distinct); display(distinctness); assert distinctness.Status.eq('PASS').all()


### 11.2 Forecast Correlation Matrix — Visualization


In [ ]:
fig,axes=plt.subplots(1,2,figsize=(14,6))
for ax,frame,protocol in [(axes[0],pa,'A'),(axes[1],pb,'B')]:
    corr=frame[MODELS].corr(); im=ax.imshow(corr,vmin=-1,vmax=1,cmap='coolwarm'); ax.set_xticks(range(len(MODELS)),MODELS,rotation=90,fontsize=7); ax.set_yticks(range(len(MODELS)),MODELS,fontsize=7); ax.set_title(f'Protocol {protocol} forecast correlation')
fig.colorbar(im,ax=axes.ravel().tolist(),shrink=.75,label='Correlation'); plt.show()


## 12. Metric Reproduction Audit

### 12.1 Independent Metric Recomputation

MAE, RMSE, MAPE, sMAPE, and MASE-48 are recomputed from each frozen vector and canonical actuals; no aggregate metrics are copied from upstream tables.

### 12.2 Metric Reproduction — Table


In [ ]:
metric_rows=[]
for protocol,frame in [('A',pa),('B',pb)]:
    for m in MODELS:
        for metric,value in metrics(frame.Actual,frame[m],scale48).items(): metric_rows.append({'Protocol':protocol,'Model':m,'Metric':metric,'Stored / Reference Value':'Recomputed from frozen vector','Recomputed Value':value,'Absolute Difference':np.nan,'Tolerance':np.nan,'Status':'RECOMPUTED'})
metric_reproduction=pd.DataFrame(metric_rows); show(metric_reproduction)


In [ ]:
# Protocol B supplies an independent horizon-level metric reference.
horizon_rows=[]
for m in MODELS:
    for h,g in pb.groupby('Horizon'):
        computed=metrics(g.Actual,g[m],scale48); reference=hm[(hm.Model==m)&(hm.Horizon==h)].iloc[0]
        for metric,value in computed.items():
            diff=abs(value-float(reference[metric])); horizon_rows.append({'Protocol':'B','Model':m,'Horizon':h,'Metric':metric,'Stored / Reference Value':float(reference[metric]),'Recomputed Value':value,'Absolute Difference':diff,'Tolerance':1e-10,'Status':'PASS' if diff<=1e-10 else 'FAIL'})
horizon_metric_audit=pd.DataFrame(horizon_rows); horizon_metric_summary=horizon_metric_audit.groupby('Model').agg(Comparisons=('Status','size'),Maximum_absolute_difference=('Absolute Difference','max'),All_pass=('Status',lambda s:s.eq('PASS').all())).reset_index(); show(horizon_metric_summary,14)
for _,r in horizon_metric_summary.iterrows(): audit_rows.append(audit_row('Metric Reproduction','Protocol B horizon reference parity','B',r.Model,'≤ 1e-10',r.Maximum_absolute_difference,'PASS' if r.All_pass else 'FAIL'))
assert horizon_metric_audit.Status.eq('PASS').all()


### 12.3 MASE-48 Denominator Audit


In [ ]:
FROZEN_SCALE=117.057971280678
mase_audit=pd.DataFrame([{'Training/pre-test source':f'{pretest.index.min()} to {pretest.index.max()}','Seasonal lag':SEASONAL_LAG,'Computed denominator':scale48,'Expected frozen denominator':FROZEN_SCALE,'Absolute difference':abs(scale48-FROZEN_SCALE),'Status':'PASS' if np.isclose(scale48,FROZEN_SCALE,atol=1e-12) else 'FAIL'}]); show(mase_audit,14)
audit_rows.append(audit_row('MASE Denominator','Pre-test seasonal scale','Both','Shared metric infrastructure',FROZEN_SCALE,scale48,mase_audit.Status.iloc[0])); assert mase_audit.Status.eq('PASS').all()


### 12.4 Ranking Reproduction — Tables


In [ ]:
rank_a=pd.DataFrame([{'Model':m,**metrics(pa.Actual,pa[m],scale48)} for m in MODELS]).sort_values('MASE_48').reset_index(drop=True); rank_a.insert(0,'Audit Rank',np.arange(1,len(rank_a)+1))
rank_b=pd.DataFrame([{'Model':m,**metrics(pb.Actual,pb[m],scale48)} for m in MODELS]).sort_values('MASE_48').reset_index(drop=True); rank_b.insert(0,'Audit Rank',np.arange(1,len(rank_b)+1))
display(Markdown('**Protocol A — AUDIT REPRODUCTION**')); show(rank_a); display(Markdown('**Protocol B — AUDIT REPRODUCTION**')); show(rank_b)


## 13. Forecast Distribution and Failure-Mode Diagnostics

### 13.1 Distribution Summary — Table


In [ ]:
distribution=[]
for protocol,frame in [('A',pa),('B',pb)]:
    a=frame.Actual.to_numpy(float)
    for m in MODELS:
        p=frame[m].to_numpy(float); distribution.append({'Protocol':protocol,'Model':m,'Mean forecast':p.mean(),'Std forecast':p.std(ddof=1),'Min':p.min(),'Max':p.max(),'Actual std':a.std(ddof=1),'Forecast/Actual std ratio':p.std(ddof=1)/a.std(ddof=1),'Residual mean':np.mean(a-p),'Constant':np.unique(p).size<=1,'Range ratio':np.ptp(p)/np.ptp(a)})
distribution_summary=pd.DataFrame(distribution); show(distribution_summary)
for _,r in distribution_summary.iterrows():
    status='PASS' if not r.Constant and r['Range ratio']>.05 else 'FAIL'; audit_rows.append(audit_row('Distribution Sanity','Nonconstant and no severe range collapse',r.Protocol,r.Model,'Range ratio > .05',r['Range ratio'],status))
assert not distribution_summary.Constant.any() and distribution_summary['Range ratio'].gt(.05).all()


### 13.2 Range/Variance Diagnostic — Visualization


In [ ]:
fig,axes=plt.subplots(2,1,figsize=(11,7))
for ax,(protocol,g) in zip(axes,distribution_summary.groupby('Protocol')):
    g=g.sort_values('Forecast/Actual std ratio'); ax.barh(g.Model,g['Forecast/Actual std ratio'],color='#0072B2'); ax.axvline(1,color='black',linestyle='--'); ax.set(title=f'Protocol {protocol}: forecast-to-actual standard-deviation ratio',xlabel='Ratio')
plt.tight_layout(); plt.show()


### 13.3 Residual Bias — Visualization

Mean residual is a gross-offset diagnostic, not robustness or significance analysis.


In [ ]:
fig,axes=plt.subplots(1,2,figsize=(13,5))
for ax,(protocol,g) in zip(axes,distribution_summary.groupby('Protocol')):
    g=g.sort_values('Residual mean'); ax.barh(g.Model,g['Residual mean'],color=np.where(g['Residual mean']>=0,'#D55E00','#0072B2')); ax.axvline(0,color='black',linewidth=.8); ax.set(title=f'Protocol {protocol}: mean residual',xlabel='Actual − forecast (MW)')
plt.tight_layout(); plt.show()


## 14. Protocol Comparability Audit

### 14.1 Why Cross-Protocol Raw Ranking Is Not a Single Competition

Protocol A and B impose different information constraints. A rank change is not evidence that one evaluation is wrong; it may show dependence on update frequency and forecast horizon. Rankings are valid within protocol, while raw errors are not a single combined competition.

### 14.2 Protocol Comparison — Table


In [ ]:
protocol_comparison=pd.DataFrame({'Property':['Forecast horizon','Actual-update access','Forecast origins','Predictions per origin','Operational interpretation','Valid within-protocol ranking?','Valid direct raw-error comparison?','Primary downstream use'],'Protocol A':['1 step','Actual revealed after each forecast',len(pa),1,'Rolling half-hour forecast','Yes','No—different information set','Rolling-performance evidence'],'Protocol B':['48 steps','None inside the horizon',pb.Origin.nunique(),48,'Genuine day-ahead forecast','Yes','No—different information set','Day-ahead-performance evidence']}); display(protocol_comparison)


### 14.3 Cross-Protocol Artifact Consistency


In [ ]:
cross_checks=[('Same target timestamps',pa.Timestamp.equals(pb.Timestamp)),('Same actual values',np.array_equal(pa.Actual,pb.Actual)),('Protocol A shape',len(pa)==len(test)),('Protocol B shape',len(pb)==len(test)),('Expected models in both',all(m in pa and m in pb for m in MODELS))]
cross_protocol_consistency=pd.DataFrame([{'Check':n,'Observed':v,'Status':'PASS' if v else 'FAIL'} for n,v in cross_checks]); display(cross_protocol_consistency)
for n,v in cross_checks:audit_rows.append(audit_row('Protocol Discipline',n,'Both','Validated artifacts',True,v,'PASS' if v else 'FAIL'))
assert cross_protocol_consistency.Status.eq('PASS').all()


## 15. Consolidated Certification Matrix

Claims about the hidden information sets of DHR-ARIMA, fitted classical models, LSTM, Chronos, and TimesFM are not directly provable from saved point vectors. They are explicitly represented below rather than silently counted as PASS.


In [ ]:
for m in CLASSICAL+['DHR_ARIMA','LSTM','Chronos_Bolt_Tiny','TimesFM']:
    audit_rows.append(audit_row('Information-Set Discipline','Internal no-leakage implementation','Both',m,'Protocol-compliant upstream construction','Not observable in final vector','NOT DIRECTLY TESTABLE'))
certification_matrix=pd.DataFrame(audit_rows); display(certification_matrix)
required=certification_matrix.Status.isin(['PASS','FAIL']); assert not certification_matrix.loc[required,'Status'].eq('FAIL').any()


### 15.1 Audit Summary — Table


In [ ]:
status_order=['PASS','FAIL','EXPECTED / DESIGN-CONSISTENT','NOT DIRECTLY TESTABLE']; counts=certification_matrix.Status.value_counts(); audit_summary=pd.DataFrame({'Status':status_order,'Count':[int(counts.get(s,0)) for s in status_order]}); audit_summary.loc[len(audit_summary)]={'Status':'TOTAL','Count':len(certification_matrix)}; display(audit_summary)


### 15.2 Certification Decision


In [ ]:
required_failures=int((certification_matrix.Status=='FAIL').sum()); certification_decision='Frozen forecast artifacts satisfy the defined artifact-level validation criteria.' if required_failures==0 else 'Frozen forecast artifacts do not satisfy all defined artifact-level validation criteria.'
display(Markdown(f'**Certification decision:** {certification_decision}'))


## 16. Visual Audit Dashboard

The high-value visual evidence is intentionally compact: Protocol B horizon coverage, foundation-model first-week alignment, protocol-specific correlation matrices, variance ratios, and mean residuals are presented in their relevant audit sections above. Together they answer completeness, identity, collapse, scale, and offset questions without duplicating upstream performance plots.


## 17. Key Findings

The findings below are generated from executed audit objects.


In [ ]:
exact_baselines=', '.join(reconstruction.groupby('Model').filter(lambda g:g.Status.eq('PASS').all()).Model.unique()); unexpected=int(distinctness['Unexpected identical pairs'].sum()); failures=int((certification_matrix.Status=='FAIL').sum())
findings=[f'All {len(artifact_inventory)} authoritative audit inputs loaded successfully with valid schemas.',f'Protocol A contains {len(pa):,} exactly aligned, finite forecast rows and passed every structural check.',f'Protocol B contains {pb.Origin.nunique():,} origins with 48 complete horizons each and passed every structural check.',f'The following deterministic baselines were independently reconstructed within tolerance in both protocols: {exact_baselines}.',f'All tested classical cross-protocol vector relationships matched their documented design; periodic-refit equality is labelled design-consistent.',f'No unexpected exact or near-exact duplicates were found across {len(pairs)} model pairs per protocol.',f'All {len(horizon_metric_audit):,} Protocol B horizon-metric comparisons matched the frozen reference within tolerance.',f'No forecast vector was constant, missing, non-finite, or below the defined severe range-collapse threshold.',certification_decision]
display(Markdown('\n'.join(f'{i+1}. {x}' for i,x in enumerate(findings))))


## 18. Limitations of the Audit

### Artifact-level validation is not source-code proof

A forecast can be correctly shaped, aligned, finite, and numerically reproducible while an upstream implementation error remains invisible in the final vector.

### Leakage verification differs by model

Deterministic baselines can be reconstructed independently. Frozen-vector analysis cannot completely prove the hidden information set used by fitted, neural, or foundation models.

### No independent model retraining

This notebook deliberately validates preserved evidence, not full end-to-end model reproducibility.

### External checkpoint reproducibility

Chronos and TimesFM checkpoint revisions were not pinned or recorded upstream, so exact external checkpoint/package state may affect future regeneration.

### Audit script trust

The notebook’s own audit implementation is not independently formally verified.

### Statistical inference not included

Certification does not establish that performance differences are statistically significant; that question belongs downstream.


## 19. Next Notebook

Next: `15_Electricity_Robustness.ipynb`

Notebook 14 certifies the frozen evidence boundary. Notebook 15 may therefore treat these forecasts as fixed inputs and investigate whether aggregate performance remains stable across demand regimes, volatility conditions, and time segments.
